In [2]:
import pandas as pd
import numpy as np

In [38]:
cpi = pd.read_csv('cpi.csv', index_col='Date', parse_dates=['Date'], dayfirst=True)
fx = pd.read_csv('fx.csv', index_col='Date', parse_dates=['Date'], dayfirst=True)
rates = pd.read_csv('rates.csv', index_col='Date', parse_dates=['Date'], dayfirst=True)

display(cpi.shape)
display(fx.shape)
display(rates.shape)

(137, 14)

(4228, 16)

(138, 14)

### Divisão dos Fatores

- Momentum: desempeho da própria moeda nos últimos 21 dias de trade (1 mês) com os últimos 62 dias (três meses)
- Fator Carry (Diferença da Taxa de Juros) -> comparar juros real com juros nominal
- Fator PPP/Taxa de Câmbio Real (câmbio + cpi)

### Momentum

In [39]:
momentum = np.log(fx.shift(21) / fx.shift(62))
momentum.dropna(inplace=True)
momentum.drop(columns='USD', inplace=True)

display(momentum.head())

,EUR,GBP,AUD,JPY,CHINA,CHF,CAD,HKD,SGD,KRW,NOK,SEK,MXN,INDIA,BRL
Date,,,,,,,,,,,,,,,
2010-03-31,0.057306,0.069748,0.008269,-0.041444,0.000088,0.042799,-0.006254,0.000889,0.002859,0.008529,0.037196,0.017623,-0.011188,-0.005633,0.007823
2010-04-01,0.047346,0.057119,0.007268,-0.033474,-0.000161,0.032175,-0.007435,0.000844,0.001688,0.001663,0.026715,0.006250,-0.008757,-0.012436,0.028938
2010-04-02,0.058615,0.063630,0.022751,-0.035218,-0.000249,0.046953,-0.000679,0.001147,0.005811,0.011522,0.044236,0.011107,-0.002515,-0.001094,0.032389
2010-04-05,0.049320,0.051579,0.010771,-0.034533,-0.000191,0.039366,-0.005042,0.000928,0.002573,0.002552,0.032680,-0.001700,-0.010723,-0.005223,0.022236
2010-04-06,0.056684,0.063255,0.018242,-0.025954,-0.000073,0.048064,-0.001847,0.000327,0.003751,0.007275,0.041840,0.004964,-0.001452,-0.002596,0.029974


### Carry

In [40]:
annual_cpi = cpi.pct_change(periods=12).dropna()

display(annual_cpi)

carry = rates.join(annual_cpi, how='inner')
display(carry.columns)

,CPIUSD,CPIEUR,CPIGBP,CPIJPY,CPICHINA,CPICHF,CPIAUD,CPICAD,CPIKRW,CPINOK,CPISEK,CPIMXN,CPIINDIA,CPIBRL
Date,,,,,,,,,,,,,,
2011-02-01,0.021249,0.024279,0.037079,-0.005269,0.050708,0.005101,0.032563,0.021626,0.039393,0.011957,0.024650,0.035723,0.088235,0.060142
2011-03-01,0.026192,0.026815,0.035794,-0.005258,0.056046,0.010043,0.032563,0.032872,0.041332,0.009730,0.029138,0.030395,0.088235,0.062990
2011-04-01,0.030772,0.028421,0.037820,-0.004206,0.054955,0.002588,0.035491,0.032759,0.038168,0.012945,0.033414,0.033607,0.094118,0.065102
2011-05-01,0.034590,0.027317,0.037736,-0.004206,0.057048,0.003690,0.035491,0.036973,0.039116,0.016269,0.033418,0.032493,0.087209,0.065528
2011-06-01,0.035023,0.027209,0.036585,-0.003161,0.066667,0.005643,0.035491,0.030981,0.042178,0.014115,0.030797,0.032765,0.086207,0.067126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-02-01,0.016698,0.009373,0.007366,-0.004985,-0.004228,-0.005041,0.011150,0.010917,0.014177,0.033273,0.013574,0.037590,0.044945,0.051954
2021-03-01,0.026678,0.013317,0.010129,-0.003988,0.003219,-0.002451,0.011150,0.021962,0.019011,0.030576,0.016860,0.046668,0.056657,0.060995
2021-04-01,0.041331,0.016219,0.016575,-0.010978,0.009226,0.003279,0.038462,0.033899,0.024925,0.029543,0.022440,0.060848,0.051399,0.067593


Index(['RATEUSD', 'RATEEUR', 'RATEGBP', 'RATEJPY', 'RATECHINA', 'RATECHF',
       'RATEAUD', 'RATECAD', 'RATEKRW', 'RATENOK', 'RATESEK', 'RATEMXN',
       'RATEINDIA', 'RATEBRL', 'CPIUSD', 'CPIEUR', 'CPIGBP', 'CPIJPY',
       'CPICHINA', 'CPICHF', 'CPIAUD', 'CPICAD', 'CPIKRW', 'CPINOK', 'CPISEK',
       'CPIMXN', 'CPIINDIA', 'CPIBRL'],
      dtype='str')

In [41]:
rate_cols = carry.columns[:14]
cpi_cols  = carry.columns[14:]

paises = ['USD','EUR','GBP','JPY','CHINA','CHF','AUD','CAD','KRW','NOK','SEK','MXN','INDIA','BRL']

real_rate = pd.DataFrame(
    carry[rate_cols].values - carry[cpi_cols].values,
    index=carry.index,
    columns=paises,
)

display(real_rate.tail())

,USD,EUR,GBP,JPY,CHINA,CHF,AUD,CAD,KRW,NOK,SEK,MXN,INDIA,BRL
Date,,,,,,,,,,,,,,
2021-02-01,0.063302,-0.550473,0.042634,-0.050015,3.254228,-0.727383,-0.001150,0.071583,0.715823,0.416727,-0.144874,4.292410,4.205055,1.948046
2021-03-01,0.043322,-0.552404,0.069871,-0.061012,3.066781,-0.727549,0.018850,0.084038,0.730989,0.349424,-0.176510,4.193332,4.193343,2.279005
2021-04-01,0.028669,-0.554369,0.073425,-0.054022,2.810774,-0.731820,0.001538,0.058601,0.715075,0.270457,-0.169150,4.189152,4.198601,2.682407
2021-05-01,0.010817,-0.559938,0.057901,-0.058006,2.823556,-0.740730,0.001538,0.061498,0.653753,0.213190,-0.170203,4.201062,4.197424,3.299439
2021-06-01,0.027044,-0.561927,0.056103,-0.067995,2.917598,-0.743772,-0.008462,0.097388,0.636532,0.171454,-0.172111,4.251214,4.194221,3.766529


## PPP

In [43]:
cpi_columns_adj = cpi.copy()
cpi_columns_adj.columns = [c.replace('CPI','') for c in cpi_columns_adj.columns]
cpi_columns_adj.index = pd.to_datetime(cpi_columns_adj.index)

display(cpi_columns_adj.tail())

,USD,EUR,GBP,JPY,CHINA,CHF,AUD,CAD,KRW,NOK,SEK,MXN,INDIA,BRL
Date,,,,,,,,,,,,,,
2021-02-01,1.213079,1.149047,1.229213,1.051634,1.302432,0.988349,1.238446,1.201557,1.187751,1.248913,1.127741,1.519863,2.016128,1.835039
2021-03-01,1.219439,1.159826,1.232584,1.052687,1.296015,0.991682,1.238446,1.207612,1.190791,1.245652,1.129503,1.532430,2.026295,1.852105
2021-04-01,1.227047,1.166467,1.240449,1.044257,1.292166,0.993751,1.247899,1.213668,1.192428,1.250000,1.132265,1.537445,2.034766,1.857846
2021-05-01,1.235189,1.169516,1.247191,1.047418,1.290884,0.996378,1.247899,1.219723,1.193246,1.248913,1.134493,1.540584,2.043237,1.873268
2021-06-01,1.245640,1.172564,1.251685,1.048472,1.285751,0.997115,1.247899,1.223184,1.193246,1.253261,1.135425,1.548793,2.061873,1.883195


In [44]:
paises_except_usd = [p for p in paises if p!='USD']

display(paises_except_usd)

['EUR',
 'GBP',
 'JPY',
 'CHINA',
 'CHF',
 'AUD',
 'CAD',
 'KRW',
 'NOK',
 'SEK',
 'MXN',
 'INDIA',
 'BRL']

In [45]:
#ALIGN IN THE SAME PERIOD -> because FX has day to day, and cpi has month to month

cpi_monthly = cpi_columns_adj.resample('ME').last()
cpi_m = cpi_monthly.copy()
cpi_m.index = cpi_m.index.to_period('M')

fx_monthly = fx.resample('ME').last()
fx_m = fx_monthly.copy()
fx_m.index = fx_m.index.to_period('M')

display(fx_m.head())
display(cpi_m.head(20))

commom_index = fx_m.index.intersection(cpi_m.index)

fx_commom = fx_m.loc[commom_index]
cpi_commom = cpi_m.loc[commom_index]



,EUR,GBP,AUD,JPY,CHINA,CHF,CAD,HKD,SGD,KRW,NOK,SEK,MXN,INDIA,BRL,USD
Date,,,,,,,,,,,,,,,,
2010-01,0.72114,0.62559,1.1308,90.269997,6.8169,1.06080,1.0700,7.76350,1.40570,1158.199951,5.9226,7.3773,13.08350,46.098000,1.8729,1.0
2010-02,0.73362,0.65621,1.1162,88.920998,6.8167,1.07330,1.0515,7.76210,1.40590,1161.400024,5.9063,7.1174,12.76840,45.900002,1.8060,1.0
2010-03,0.74030,0.65860,1.0922,93.519997,6.8159,1.05366,1.0156,7.76380,1.39922,1131.599976,5.9395,7.2157,12.35530,44.799999,1.7805,1.0
2010-04,0.75199,0.65359,1.0813,93.830002,6.8160,1.07560,1.0180,7.76350,1.37000,1107.900024,5.9030,7.2442,12.30180,44.250000,1.7287,1.0
2010-05,0.81381,0.68861,1.1812,91.190002,6.8181,1.15630,1.0431,7.78704,1.39959,1200.000000,6.4611,7.8217,12.90679,46.358002,1.8144,1.0


,USD,EUR,GBP,JPY,CHINA,CHF,AUD,CAD,KRW,NOK,SEK,MXN,INDIA,BRL
Date,,,,,,,,,,,,,,
2010-02,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2010-03,1.000331,1.010996,1.004494,1.002108,0.992949,1.001370,1.000000,1.000000,1.002023,1.005435,1.002395,1.007099,1.000000,1.005199
2010-04,1.000561,1.015133,1.010112,1.002108,0.994975,1.009929,1.006302,1.003460,1.006057,1.007609,1.002528,1.003890,1.000000,1.010930
2010-05,1.000041,1.016331,1.012360,1.002108,0.994002,1.008887,1.006302,1.006055,1.007074,1.002174,1.004391,0.997565,1.011765,1.015278
2010-06,0.999623,1.016331,1.013483,1.000000,0.988004,1.004494,1.006302,1.005190,1.006057,1.001087,1.004557,0.997252,1.023529,1.015278
2010-07,1.001491,1.011976,1.011236,0.994731,0.991976,0.997155,1.013655,1.010381,1.008080,0.995652,1.001464,0.999418,1.047059,1.015379
2010-08,1.002955,1.013936,1.015730,0.995785,0.997893,0.997016,1.013655,1.009515,1.013143,0.993478,1.001530,1.002194,1.047059,1.015784
2010-09,1.004575,1.017093,1.015730,0.997893,1.003890,0.996785,1.013655,1.011246,1.021211,1.000000,1.009980,1.007447,1.052941,1.020353
2010-10,1.008072,1.020250,1.017978,1.001054,1.010942,1.002089,1.017857,1.015571,1.021211,1.001087,1.013207,1.013666,1.064706,1.028007


### rer -> real exchange rate

$$RER = Spot * \frac{CPI_{usa}}{CPI_{loc}}$$
$$ ln(RER)=ln(Spot*\frac{CPI_{usa}}{CPI_{loc}}) $$
$$ ln(RER)=ln(Spot)+ln(CPI_{usa})-ln(CPI_{loc}) $$

prop log simples

In [49]:
ppp_signals = pd.DataFrame(index=commom_index, columns=paises_except_usd, dtype=float)

for moeda in paises_except_usd:
    if moeda in fx_m.columns and moeda in cpi_m.columns:
        log_spot = np.log(fx_m[moeda])
        log_price_diff = np.log(cpi_m['USD']) - np.log(cpi_m[moeda])
        rer = log_spot + log_price_diff
        ppp_signals[moeda] = rer - rer.expanding().mean()

ppp_signals.dropna(inplace=True)
display(ppp_signals.head(10))

,EUR,GBP,JPY,CHINA,CHF,AUD,CAD,KRW,NOK,SEK,MXN,INDIA,BRL
Date,,,,,,,,,,,,,
2010-02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2010-03,-0.000770,-0.000259,0.024326,0.003645,-0.009753,-0.010702,-0.017203,-0.013842,0.000258,0.005828,-0.019816,-0.011963,-0.009537
2010-04,0.007362,-0.008828,0.018577,0.001234,0.001717,-0.017857,-0.012045,-0.025864,-0.005224,0.006578,-0.013822,-0.016057,-0.029678
2010-05,0.063500,0.030468,-0.007862,0.001500,0.055932,0.052493,0.006907,0.039346,0.067503,0.060677,0.029990,0.013699,0.010422
2010-06,0.054241,0.000219,-0.029527,0.000279,-0.008341,0.046655,0.020696,0.049881,0.061819,0.044270,0.025326,0.002100,-0.002427
2010-07,-0.003040,-0.036881,-0.037615,-0.002391,-0.028394,-0.025879,-0.012103,0.011656,-0.001876,-0.023100,0.001509,-0.016353,-0.015763
2010-08,0.021870,-0.021946,-0.051029,-0.002342,-0.035962,-0.008922,0.016879,0.013439,0.033254,0.004043,0.038319,-0.003332,-0.014309
2010-09,-0.046236,-0.036253,-0.053053,-0.020807,-0.072206,-0.079350,-0.008903,-0.034219,-0.039567,-0.087799,-0.011285,-0.048136,-0.041096
2010-10,-0.050496,-0.029879,-0.069307,-0.021880,-0.053985,-0.074228,-0.011863,-0.032649,-0.025968,-0.069512,-0.030379,-0.054961,-0.034880



### prove -> ppp EUR marc (2010-03)

rerfev = ln(0.73362)+ln(1.0000)-ln(1.0000)
rerfev = -0.309764

rermarc = ln(0.74030)+ln(1.000331)-ln(1.010996)
rermarc = -0.311304807

mean = -0.309764-0.311304807/2 = -0.3105344
signal = rermarc - mean = -0.311304807-(-0.3105344) = **-0.00007704** 

---

given that the fx rate went from 0.733 (fev) to 0.74 (marc) -> -0.9%, and inflation from 1.000000 to 1.010996 -> 1.1%
i expected it to fall 1.1% but it only fell 0.9%

rer = -0.000770 -> sell euros


### Volatilidade Implicita e Volatilidade Realizada

### Backtest